[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/padillacm/IA-practice/blob/master/tokenization-embeddings-course/06_tokenizer_free_bytes_blt/06_proyecto_tokenizer_free_bytes_blt.ipynb)

# Proyecto 06 · Mini *patcher* entrópico vs BPE

| | |
|---|---|
| **Nivel** | 🔴 Experto |
| **Duración estimada** | ≈ 2 h |
| **GPU recomendada** | Ninguna (todo en CPU con `FAST_DEV_RUN=True`, ≈ 2-3 min) |
| **Prerrequisitos** | Lección 06 (entropía del siguiente byte, parches, parcheo incremental), módulos 02-03 (BPE con `tokenizers`) |

## Contexto de negocio

Eres ML engineer en **Políglota Labs**. El CTO duda entre mantener el tokenizer BPE que entrenasteis (módulos 02-04) o explorar una ruta «sin vocabulario» para el futuro modelo generativo del
producto. Te pide una **nota técnica con números**: construye un *patcher* por entropía con un modelo de bytes diminuto entrenado en el corpus Políglota y compáralo con BPE en
(i) tamaño medio de unidad, (ii) **propiedad incremental**, (iii) **robustez a erratas** y (iv) **coste de FLOPs**. Con honestidad: es un banco diminuto, no el resultado del paper.

**Conexión.** Módulo anterior (05): glitch tokens y fragilidad ante erratas → aquí mides la fragilidad de la *segmentación*. Módulo siguiente (07): la frontera se **aprende** (H-Net).

## Dataset
Corpus Políglota vía `carga_corpus()` (Wikipedia ES/EN + CodeSearchNet; fallback sintético + funciones reales de la stdlib). Si imprime «SINTÉTICO», los umbrales de la rúbrica (calibrados sobre datos sintéticos) siguen siendo orientativos pero los valores absolutos cambiarán.

## Entregables y criterios de evaluación

1. Entropía de bytes (n-grama de orden 4 ya dado) y **reglas de parcheo** global y monótona (TODO 1-3).
2. Prueba de **parcheo incremental** para entropía y BPE (TODO 4).
3. Métricas de **robustez a erratas** (inflación y estabilidad) para ambos (TODO 5).
4. **Coste**: parche medio necesario para ahorrar un 25 % de FLOPs/byte frente a BPE (TODO 6).
5. Una **conclusión** escrita en la variable `CONCLUSION` (≥ 300 caracteres) que diga qué recomendarías a Políglota y qué no puedes concluir.

| Criterio | Umbral (calibrado con la solución de referencia) |
|---|---|
| R1 · tamaño medio de parche en validación (global y monótono) | objetivo 4.5 B/parche ± 10 % (medido: 4.27 y 4.48) |
| R2 · parcheo incremental | entropía = 100 %, BPE < 100 % (medido: 100 % y 92.7 %) |
| R3 · inflación de unidades con 5 % de erratas (entropía) | ≤ 1.1 (medido: 1.036; BPE 1.005) |
| R4 · estabilidad de palabras intactas (entropía) | ≥ 0.95 (medido: 0.993) |
| R5 · estabilidad de palabras intactas (BPE) | ≥ 0.98 (control de que tu métrica está bien; medido: 1.000) |
| R6 · parche medio para −25 % de FLOPs/byte frente a un BPE industrial (4.4 B/token) | ≤ 6.5 bytes (medido: 6.0) |
| R7 · conclusión escrita | ≥ 300 caracteres |

> Los umbrales salen de lo **medido** por la solución en `FAST_DEV_RUN=True` con corpus sintético (ver celda de rúbrica al final); con datos reales vuelve a medir y ajusta.

In [ ]:
!pip install -q torch numpy matplotlib tokenizers transformers datasets

In [ ]:
import os, sys, time, math, random, warnings
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
warnings.filterwarnings("ignore")
import numpy as np
import torch

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
FAST_DEV_RUN = True      # True: CPU, minutos. False: GPU (T4/L4), escala completa
device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cpu":
    torch.set_num_threads(int(os.environ.get("OMP_NUM_THREADS", min(4, os.cpu_count() or 1))))
print("device:", device, "| FAST_DEV_RUN:", FAST_DEV_RUN, "| torch", torch.__version__)
import re

import re
import matplotlib.pyplot as plt
import torch.nn as nn, torch.nn.functional as F
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": .25,
                     "axes.spines.top": False, "axes.spines.right": False})
AZUL, NARANJA, VERDE, GRIS = "#1F6FEB", "#F28E2B", "#2CA02C", "#6B7280"

In [ ]:
# ── Datos para el corpus SINTÉTICO de Políglota (último recurso si no hay red / Hugging Face) ──
# (artículo+concepto ES, artículo+concepto EN, definición ES, definición EN)
CONCEPTOS = [
    ("el índice invertido", "the inverted index", "una estructura que asocia cada término con la lista de documentos que lo contienen", "a structure that maps each term to the list of documents that contain it"),
    ("la caché", "the cache", "una memoria rápida que guarda resultados recientes para no recalcularlos", "a fast memory that stores recent results so they are not recomputed"),
    ("la base de datos relacional", "the relational database", "un sistema que organiza los datos en tablas relacionadas mediante claves", "a system that organizes data into tables related through keys"),
    ("el balanceador de carga", "the load balancer", "un componente que reparte las peticiones entre varios servidores", "a component that spreads requests across several servers"),
    ("el contenedor Docker", "the Docker container", "un proceso aislado que empaqueta una aplicación con sus dependencias", "an isolated process that packages an application with its dependencies"),
    ("el control de versiones", "version control", "un sistema que registra los cambios de los archivos a lo largo del tiempo", "a system that records changes to files over time"),
    ("la API REST", "the REST API", "una interfaz que expone recursos mediante URL y verbos HTTP", "an interface that exposes resources through URLs and HTTP verbs"),
    ("la autenticación con tokens", "token authentication", "un mecanismo que identifica al usuario con un testigo firmado en cada petición", "a mechanism that identifies the user with a signed token on every request"),
    ("la expresión regular", "the regular expression", "un patrón que describe un conjunto de cadenas de texto", "a pattern that describes a set of text strings"),
    ("el hilo de ejecución", "the thread", "una secuencia de instrucciones que comparte memoria con otros hilos del proceso", "a sequence of instructions that shares memory with other threads of the process"),
    ("el recolector de basura", "the garbage collector", "un mecanismo que libera automáticamente la memoria que ya no se usa", "a mechanism that automatically frees memory that is no longer used"),
    ("la cola de mensajes", "the message queue", "un búfer que desacopla a los productores de los consumidores de eventos", "a buffer that decouples event producers from consumers"),
    ("la tabla hash", "the hash table", "una estructura que guarda pares clave-valor y busca en tiempo casi constante", "a structure that stores key-value pairs and looks them up in near-constant time"),
    ("el árbol binario de búsqueda", "the binary search tree", "un árbol ordenado donde cada nodo separa los valores menores de los mayores", "an ordered tree where each node separates smaller values from larger ones"),
    ("la red neuronal", "the neural network", "un modelo formado por capas de unidades que aprenden pesos a partir de ejemplos", "a model made of layers of units that learn weights from examples"),
    ("el descenso de gradiente", "gradient descent", "un método que ajusta los parámetros en la dirección que reduce la pérdida", "a method that adjusts parameters in the direction that reduces the loss"),
    ("el sobreajuste", "overfitting", "el fenómeno por el que un modelo memoriza el entrenamiento y generaliza mal", "the phenomenon where a model memorizes the training set and generalizes poorly"),
    ("la validación cruzada", "cross-validation", "una técnica que estima el rendimiento repartiendo los datos en varios pliegues", "a technique that estimates performance by splitting the data into several folds"),
    ("el tokenizador", "the tokenizer", "el componente que convierte el texto en una secuencia de identificadores enteros", "the component that turns text into a sequence of integer identifiers"),
    ("la ventana de contexto", "the context window", "la cantidad máxima de tokens que un modelo puede leer a la vez", "the maximum number of tokens a model can read at once"),
    ("el compilador", "the compiler", "un programa que traduce código fuente a instrucciones de máquina", "a program that translates source code into machine instructions"),
    ("el sistema de archivos", "the file system", "la capa que organiza cómo se guardan y se nombran los datos en disco", "the layer that organizes how data is stored and named on disk"),
    ("la memoria virtual", "virtual memory", "una abstracción que da a cada proceso su propio espacio de direcciones", "an abstraction that gives each process its own address space"),
    ("el protocolo TCP", "the TCP protocol", "un protocolo que entrega bytes de forma ordenada y fiable entre dos máquinas", "a protocol that delivers bytes reliably and in order between two machines"),
    ("el cifrado simétrico", "symmetric encryption", "un esquema que usa la misma clave para cifrar y descifrar", "a scheme that uses the same key to encrypt and decrypt"),
    ("las pruebas unitarias", "unit tests", "comprobaciones automáticas que verifican una función aislada", "automatic checks that verify a single isolated function"),
    ("la integración continua", "continuous integration", "una práctica que compila y prueba cada cambio de forma automática", "a practice that builds and tests every change automatically"),
    ("los microservicios", "microservices", "una arquitectura que divide la aplicación en servicios pequeños e independientes", "an architecture that splits the application into small independent services"),
    ("la paginación", "pagination", "una técnica que devuelve los resultados en bloques de tamaño fijo", "a technique that returns results in fixed-size blocks"),
    ("la limitación de tasa", "rate limiting", "una política que restringe cuántas peticiones puede hacer un cliente por minuto", "a policy that restricts how many requests a client can make per minute"),
    ("la serialización JSON", "JSON serialization", "el proceso de convertir objetos en texto JSON y viceversa", "the process of converting objects to JSON text and back"),
    ("la variable de entorno", "the environment variable", "un par nombre-valor que el sistema operativo entrega a cada proceso", "a name-value pair that the operating system hands to each process"),
    ("el entorno virtual", "the virtual environment", "una carpeta aislada con su propio intérprete de Python y sus paquetes", "an isolated folder with its own Python interpreter and packages"),
    ("el decorador", "the decorator", "una función que envuelve a otra para añadirle comportamiento", "a function that wraps another one to add behavior"),
    ("el generador", "the generator", "una función que produce valores perezosamente con la palabra yield", "a function that lazily produces values with the yield keyword"),
    ("el manejo de excepciones", "exception handling", "el mecanismo para capturar errores sin que el programa se detenga", "the mechanism for catching errors without stopping the program"),
    ("la programación asíncrona", "asynchronous programming", "un modelo que solapa esperas de entrada y salida sin bloquear el hilo", "a model that overlaps input and output waits without blocking the thread"),
    ("el bloqueo de base de datos", "the database lock", "un candado que impide que dos transacciones modifiquen la misma fila a la vez", "a lock that prevents two transactions from modifying the same row at once"),
    ("la migración de esquema", "the schema migration", "un script versionado que cambia la estructura de la base de datos", "a versioned script that changes the structure of the database"),
    ("el registro de eventos", "event logging", "la práctica de escribir mensajes con marca de tiempo sobre lo que hace el sistema", "the practice of writing timestamped messages about what the system does"),
]
HERRAMIENTAS = ["PostgreSQL", "Redis", "nginx", "Docker", "pytest", "Kubernetes", "FastAPI", "SQLite", "Git", "Linux", "Celery", "Prometheus"]
PARAMETROS = ["max_connections", "timeout_seconds", "cache_size", "batch_size", "log_level", "retry_count", "pool_size", "num_workers"]
# aspecto -> (título ES, título EN, palabra clave ES, palabra clave EN, plantillas ES, plantillas EN)
ASPECTOS = {
    "intro": ("{n}: qué es y para qué sirve", "{n}: what it is and what it is for", "qué es", "what is",
        ["Se usa en {t} para simplificar el trabajo diario del equipo.", "Entender {c} es un requisito antes de pasar a producción.", "Su idea central es sencilla y aparece en casi todos los sistemas modernos.", "La documentación de {t} dedica un capítulo completo a {c}."],
        ["It is used in {t} to simplify the daily work of the team.", "Understanding {c} is a prerequisite before going to production.", "Its core idea is simple and appears in almost every modern system.", "The {t} documentation devotes a whole chapter to {c}."]),
    "config": ("{n}: cómo configurarlo", "{n}: how to configure it", "configurar", "configure",
        ["Para configurar {c} edita el archivo de ajustes y reinicia {t}.", "El parámetro {p} controla el comportamiento de {c} y su valor por defecto es {x}.", "Después de cambiar {p} a {x}, comprueba el resultado con un comando de diagnóstico.", "Guarda la configuración de {c} en un archivo versionado y no en la consola."],
        ["To configure {c}, edit the settings file and restart {t}.", "The {p} parameter controls the behavior of {c} and its default value is {x}.", "After changing {p} to {x}, check the result with a diagnostic command.", "Keep the configuration of {c} in a versioned file and not in the console."]),
    "errores": ("{n}: errores frecuentes", "{n}: common errors", "errores", "errors",
        ["Un error frecuente con {c} es olvidar el valor de {p} y dejarlo en {x}.", "Si aparece un fallo intermitente en {t}, revisa primero los registros de {c}.", "Muchos equipos descubren el problema de {c} solo cuando la carga se multiplica por {x}.", "El mensaje de error suele ser engañoso: la causa real está en {c}, no en {t}."],
        ["A common mistake with {c} is forgetting {p} and leaving it at {x}.", "If an intermittent failure shows up in {t}, check the logs of {c} first.", "Many teams only discover the problem with {c} when the load grows by {x} times.", "The error message is usually misleading: the real cause is in {c}, not in {t}."]),
    "rendimiento": ("{n}: rendimiento y optimización", "{n}: performance and optimization", "rendimiento", "performance",
        ["Para medir el rendimiento de {c} usa un perfilador y no tu intuición.", "Subir {p} a {x} mejora la latencia, pero consume más memoria en {t}.", "La optimización prematura de {c} suele complicar el código sin ganar tiempo.", "En una prueba con {x} usuarios, {c} redujo la latencia mediana a la mitad."],
        ["To measure the performance of {c}, use a profiler and not your intuition.", "Raising {p} to {x} improves latency but uses more memory in {t}.", "Premature optimization of {c} usually complicates the code without saving time.", "In a test with {x} users, {c} cut the median latency in half."]),
    "practicas": ("{n}: buenas prácticas", "{n}: best practices", "buenas prácticas", "best practices",
        ["Una buena práctica con {c} es documentar {p} y automatizar su despliegue en {t}.", "Revisa {c} en cada revisión de código y escribe pruebas para los casos límite.", "Mantén {c} simple: añade complejidad solo cuando los datos lo justifiquen.", "Define alertas sobre {c} antes del primer incidente, no después."],
        ["A good practice with {c} is to document {p} and automate its deployment in {t}.", "Review {c} in every code review and write tests for the edge cases.", "Keep {c} simple: add complexity only when the data justifies it.", "Define alerts on {c} before the first incident, not after."]),
    "ejemplo": ("{n}: ejemplo en Python", "{n}: Python example", "ejemplo", "example",
        ["En este ejemplo en Python usamos {t} para mostrar {c} paso a paso.", "El script define {p} = {x} y luego imprime el resultado de {c}.", "Ejecuta el ejemplo con Python 3.11 y compara la salida con la esperada.", "Puedes adaptar el código de {c} a tu propio proyecto cambiando solo {p}."],
        ["In this Python example we use {t} to show {c} step by step.", "The script defines {p} = {x} and then prints the result of {c}.", "Run the example with Python 3.11 and compare the output with the expected one.", "You can adapt the code for {c} to your own project by changing only {p}."]),
    "seguridad": ("{n}: consideraciones de seguridad", "{n}: security considerations", "seguridad", "security",
        ["Desde el punto de vista de la seguridad, {c} nunca debe exponerse sin autenticación.", "Rota las credenciales de {t} cada {x} días y no las guardes en el repositorio.", "Un ataque contra {c} suele aprovechar el valor de {p} mal configurado.", "Audita los accesos a {c} y conserva los registros durante {x} días."],
        ["From a security standpoint, {c} must never be exposed without authentication.", "Rotate the {t} credentials every {x} days and do not store them in the repository.", "An attack on {c} usually exploits a misconfigured {p} value.", "Audit access to {c} and keep the logs for {x} days."]),
}

In [ ]:
# ── Generador del corpus sintético (mismo esquema que el real) ────────────────
import random


def _es_fix(s: str) -> str:
    """Contracciones del español: 'a el' -> 'al', 'de el' -> 'del'."""
    return s.replace(" a el ", " al ").replace(" de el ", " del ").replace("De el ", "Del ").replace("A el ", "Al ")


def _sintetico_wiki(lang: str, n_docs: int, n_queries: int, seed: int = 42):
    """Docs técnicos sintéticos ES/EN: concepto x aspecto x herramienta. Devuelve (docs, queries, qrels)."""
    rng = random.Random(seed + (0 if lang == "es" else 1))
    li = 0 if lang == "es" else 1
    combos = [(c, a, t) for c in range(len(CONCEPTOS)) for a in ASPECTOS for t in range(len(HERRAMIENTAS))]
    rng.shuffle(combos)
    docs, titles = [], []
    for k, (ci, asp, ti) in enumerate(combos[:n_docs]):
        concepto, definicion = CONCEPTOS[ci][li], CONCEPTOS[ci][2 + li]
        tool = HERRAMIENTAS[ti]
        tit_t, plantillas = ASPECTOS[asp][li], ASPECTOS[asp][4 + li]
        nombre = concepto.split(" ", 1)[1] if concepto.split(" ", 1)[0] in ("el", "la", "los", "las", "the") else concepto
        slots = lambda: dict(c=concepto, t=tool, p=rng.choice(PARAMETROS), x=rng.choice([8, 16, 30, 64, 100, 256, 1024]))
        frases = [("{C} " + ("es " if li == 0 else "is ") + definicion + ".").format(C=concepto)]
        frases += [pl.format(**slots()) for pl in rng.sample(plantillas, len(plantillas))]
        frases.append(rng.choice(ASPECTOS["intro"][4 + li]).format(**slots()))
        for _ in range(2):  # ruido: frases de otro concepto (hace el retrieval no trivial)
            otro = rng.randrange(len(CONCEPTOS))
            otra = rng.choice(list(ASPECTOS.values()))[4 + li]
            s = slots(); s["c"] = CONCEPTOS[otro][li]
            frases.append(rng.choice(otra).format(**s))
        s = slots()
        frases.append(("Ejemplo: `{p} = {x}`." if li == 0 else "Example: `{p} = {x}`.").format(**s))
        text = " ".join(f[0].upper() + f[1:] for f in frases)
        docs.append({"id": f"{lang}-syn-{k:04d}", "lang": lang, "text": _es_fix(text) if lang == "es" else text})
        titles.append((nombre, ASPECTOS[asp][2 + li], tool, tit_t.format(n=nombre) + f" ({tool})"))
    queries, qrels = [], {}
    for j, k in enumerate(rng.sample(range(len(docs)), min(n_queries, len(docs)))):
        nombre, kw, tool, titulo = titles[k]
        q = titulo.replace(":", "").replace("(", "").replace(")", "") if j % 2 == 0 else f"{nombre} {kw} {tool}"
        qid = f"{lang}-q{j:04d}"
        queries.append({"qid": qid, "lang": lang, "text": q})
        qrels[qid] = {docs[k]["id"]: 1}
    return docs, queries, qrels

In [ ]:
# ── carga_corpus(): cadena oficial (HF) -> espejo (API de Wikipedia / stdlib) -> sintético ─────
import ast
import json
import os
import random
import urllib.parse
import urllib.request

SEED = 42


def _strip_docstring(func_src: str):
    """Devuelve (código sin docstring, primera frase del docstring) o None si no hay docstring útil."""
    try:
        fn = ast.parse(func_src).body[0]
        doc = ast.get_docstring(fn)
        first = fn.body[0]
    except Exception:
        return None
    if not doc or not (isinstance(first, ast.Expr) and isinstance(getattr(first, "value", None), ast.Constant)):
        return None
    lines = func_src.splitlines()
    code = "\n".join(lines[: first.lineno - 1] + lines[first.end_lineno :])
    frase = doc.strip().split("\n\n")[0].replace("\n", " ").strip()
    frase = frase.split(". ")[0].rstrip(".")
    return (code, frase) if len(frase.split()) >= 4 else None


def _code_stdlib(n_docs: int, n_queries: int, rng: random.Random):
    """Espejo OFFLINE con código real: funciones de la biblioteca estándar con su docstring como query."""
    import importlib, inspect, textwrap
    mods = ["argparse", "bisect", "calendar", "collections", "colorsys", "copy", "csv", "difflib", "fnmatch", "fractions",
            "functools", "getpass", "glob", "heapq", "html", "json", "operator", "os", "pathlib", "posixpath", "queue",
            "random", "re", "shlex", "shutil", "statistics", "string", "tempfile", "textwrap", "urllib.parse", "uuid", "zipfile"]
    cands = []
    for name in mods:
        try:
            src = inspect.getsource(importlib.import_module(name))
            tree = ast.parse(src)
        except Exception:
            continue
        for node in ast.walk(tree):
            if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and node.end_lineno - node.lineno >= 4:
                seg = textwrap.dedent("\n".join(src.splitlines()[node.lineno - 1 : node.end_lineno]))
                r = _strip_docstring(seg)
                if r and 200 <= len(r[0]) <= 2500:
                    cands.append((f"{name}.{node.name}", r[0], r[1]))
    rng.shuffle(cands)
    return cands[:n_docs]


def _code_hf(n_docs: int, rng: random.Random):
    """Oficial: CodeSearchNet (Python) en streaming. Prueba varios repos por cambios de formato en `datasets`."""
    from datasets import load_dataset
    errs = []
    for repo, kw in [("code-search-net/code_search_net", {"name": "python"}), ("code_search_net", {"name": "python"}),
                     ("claudios/code_search_net", {"name": "python"}), ("Nan-Do/code-search-net-python", {})]:
        try:
            ds = load_dataset(repo, split="train", streaming=True, **kw).shuffle(seed=SEED, buffer_size=2000)
            out = []
            for ex in ds:
                src = ex.get("func_code_string") or ex.get("whole_func_string") or ex.get("code") or ""
                r = _strip_docstring(src) if '"""' in src or "'''" in src else None
                doc = (ex.get("func_documentation_string") or ex.get("docstring") or "").strip().split("\n\n")[0]
                doc = doc.replace("\n", " ").split(". ")[0].rstrip(".")
                code = r[0] if r else src
                if 200 <= len(code) <= 2500 and len(doc.split()) >= 4:
                    out.append((ex.get("func_name") or f"csn.{len(out)}", code, doc))
                if len(out) >= n_docs:
                    return out
        except Exception as e:  # noqa: BLE001
            errs.append(f"{repo}: {type(e).__name__}")
    raise RuntimeError("CodeSearchNet no disponible -> " + "; ".join(errs))


def _wiki_hf(lang: str, n_docs: int, cap: int):
    """Oficial: wikimedia/wikipedia (snapshot 20231101) en streaming con un buffer pequeño."""
    from datasets import load_dataset
    ds = load_dataset("wikimedia/wikipedia", f"20231101.{lang}", split="train", streaming=True)
    ds = ds.shuffle(seed=SEED, buffer_size=2000)
    out = []
    for ex in ds:
        if len(ex["text"]) >= 600 and len(ex["title"]) >= 3:
            out.append((f"{lang}-{ex['id']}", ex["title"], ex["text"][:cap]))
        if len(out) >= n_docs:
            break
    if len(out) < n_docs // 2:
        raise RuntimeError("pocos artículos")
    return out


def _wiki_api(lang: str, n_docs: int):
    """Espejo: API pública de Wikipedia (artículos aleatorios, solo la sección introductoria; ~20 por petición)."""
    out, seen, tries = [], set(), 0
    while len(out) < n_docs and tries < 80:
        tries += 1
        qs = urllib.parse.urlencode({"action": "query", "format": "json", "generator": "random", "grnnamespace": 0,
                                     "grnlimit": 20, "prop": "extracts", "exintro": 1, "explaintext": 1, "exlimit": 20})
        req = urllib.request.Request(f"https://{lang}.wikipedia.org/w/api.php?{qs}",
                                     headers={"User-Agent": "poliglota-course/1.0 (curso educativo)"})
        with urllib.request.urlopen(req, timeout=20) as r:
            pages = json.load(r).get("query", {}).get("pages", {})
        for p in pages.values():
            if p["pageid"] not in seen and len(p.get("extract", "")) >= 300:
                seen.add(p["pageid"])
                out.append((f"{lang}-{p['pageid']}", p["title"], p["extract"]))
    if len(out) < n_docs // 2:
        raise RuntimeError("pocos artículos")
    return out[:n_docs]


def carga_corpus(fast_dev: bool = True) -> dict:
    """Corpus Políglota -> {"docs": [{id, lang, text}], "queries": [{qid, lang, text}], "qrels": {qid: {doc_id: 1}},
    "source": "real" | "synthetic"}. Cadena: oficial (HF) -> espejo (API Wikipedia; código de la stdlib) -> sintético."""
    rng = random.Random(SEED)
    n_wiki, n_code, q_wiki, q_code, cap = (150, 100, 40, 30, 1500) if fast_dev else (3000, 1500, 200, 150, 6000)
    docs, queries, qrels, source = [], [], {}, "real"
    wiki = {}
    for lang in ("es", "en"):
        for nombre, fn in (("HF wikimedia/wikipedia", lambda: _wiki_hf(lang, n_wiki, cap)),
                           ("API de Wikipedia (espejo)", lambda: _wiki_api(lang, min(n_wiki, 400)))):
            try:
                wiki[lang] = fn()
                print(f"[carga_corpus] Wikipedia {lang}: {len(wiki[lang])} artículos desde {nombre}")
                break
            except Exception as e:  # noqa: BLE001
                print(f"[carga_corpus] {nombre} ({lang}) falló: {type(e).__name__}: {str(e)[:80]}")
    if len(wiki) == 2:
        for lang, arts in wiki.items():
            docs += [{"id": i, "lang": lang, "text": t} for i, _, t in arts]
            for j, k in enumerate(rng.sample(range(len(arts)), min(q_wiki, len(arts)))):
                qid = f"{lang}-q{j:04d}"
                queries.append({"qid": qid, "lang": lang, "text": arts[k][1]})
                qrels[qid] = {arts[k][0]: 1}
    else:
        source = "synthetic"
        print("⚠️  [carga_corpus] SIN acceso a Wikipedia: uso texto ES/EN SINTÉTICO (mismo esquema). Los números NO son comparables con datos reales.")
        for lang in ("es", "en"):
            d, q, r = _sintetico_wiki(lang, n_wiki, q_wiki, SEED)
            docs += d; queries += q; qrels.update(r)
    try:
        code = _code_hf(n_code, rng)
        print(f"[carga_corpus] código: {len(code)} funciones desde CodeSearchNet (HF)")
    except Exception as e:  # noqa: BLE001
        print(f"[carga_corpus] CodeSearchNet falló ({str(e)[:90]}); uso funciones REALES de la stdlib de Python (espejo offline)")
        code = _code_stdlib(n_code, q_code, rng)
    for k, (name, src, doc) in enumerate(code):
        docs.append({"id": f"code-{k:04d}", "lang": "code", "text": src})
    for j, k in enumerate(rng.sample(range(len(code)), min(q_code, len(code)))):
        qid = f"code-q{j:04d}"
        queries.append({"qid": qid, "lang": "code", "text": code[k][2]})
        qrels[qid] = {f"code-{k:04d}": 1}
    return {"docs": docs, "queries": queries, "qrels": qrels, "source": source}

## Paso 0 · Datos y utilidades (dados)

Partimos el corpus en entrenamiento/validación por documento, entrenamos un BPE de 512 tokens y definimos las reglas de segmentación «sin modelo» de la lección.

In [ ]:
corp = carga_corpus(FAST_DEV_RUN)
print("fuente del corpus:", corp["source"], "| docs:", len(corp["docs"]))
rng = random.Random(SEED)
docs = corp["docs"][:]; rng.shuffle(docs)
cut = int(0.9 * len(docs))
DOCS_TR, DOCS_VA = docs[:cut], docs[cut:]
MAX_CHARS = 1500 if FAST_DEV_RUN else 6000
LIMITE_BYTES = None if FAST_DEV_RUN else 3_000_000        # el modelo n-grama es Python puro: acota el modo completo

def texto(ds):
    return "\n\n".join(d["text"][:MAX_CHARS] for d in ds)

TXT_TR, TXT_VA = texto(DOCS_TR), texto(DOCS_VA)
TR = np.frombuffer(TXT_TR.encode("utf-8"), dtype=np.uint8).copy()[:LIMITE_BYTES]
VA = np.frombuffer(TXT_VA.encode("utf-8"), dtype=np.uint8).copy()[:LIMITE_BYTES]
print(f"bytes train={len(TR):,} | val={len(VA):,}")
for lang in ["es", "en", "code"]:
    t = "".join(d["text"][:MAX_CHARS] for d in DOCS_TR if d["lang"] == lang)
    print(f"{lang:>4}: {len(t):>8,} chars -> {len(t.encode()):>8,} bytes  ({len(t.encode())/max(1, len(t)):.3f} bytes/char)")

In [ ]:
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders

def entrena_bpe(docs_, vocab_size):
    tk = Tokenizer(models.BPE())
    tk.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
    tk.decoder = decoders.ByteLevel()
    trn = trainers.BpeTrainer(vocab_size=vocab_size, initial_alphabet=pre_tokenizers.ByteLevel.alphabet(),
                              show_progress=False)
    tk.train_from_iterator((d["text"][:MAX_CHARS] for d in docs_), trn)
    return tk

def bytes_por_token(tk, textos):
    n_b = sum(len(t.encode()) for t in textos)
    return n_b / sum(len(tk.encode(t).ids) for t in textos)


VOCABS = [300, 512, 1024, 2048] if FAST_DEV_RUN else [512, 4096, 16384, 32000]
TK = entrena_bpe(DOCS_TR, VOCABS[1])
print('BPE entrenado, vocab =', TK.get_vocab_size())

In [ ]:
MUESTRA = "El servidor procesa las peticiones entrantes si falta la variable de entorno."

def inicios_stride(n, k=4):
    s = np.zeros(n, bool); s[::k] = True; return s

def es_espacio(b):                      # byte "tipo espacio": ni letra latina, ni dígito, ni continuación UTF-8
    return not (48 <= b <= 57 or 65 <= b <= 90 or 97 <= b <= 122 or 0x80 <= b <= 0xBF or b >= 0xC0)

def inicios_espacio(data):             # SpaceByte: parche nuevo tras un byte-espacio (cada parche con >=1 byte no-espacio)
    s = np.zeros(len(data), bool); s[0] = True
    for i in range(1, len(data)):
        s[i] = es_espacio(int(data[i - 1])) and not es_espacio(int(data[i]))
    return s

def inicios_bpe(tk, txt):
    off = np.cumsum([0] + [len(c.encode()) for c in txt])        # offset en caracteres -> en bytes
    s = np.zeros(off[-1], bool)
    for a, _ in tk.encode(txt).offsets:
        s[off[a]] = True
    return s

def marca(txt_bytes, starts):
    return "".join(("|" if s and i else "") + bytes([c]).decode("latin-1") for i, (c, s) in enumerate(zip(txt_bytes, starts)))

In [ ]:
def flops_capa(d, ctx, ffn=4):
    """FLOPs (forward) de UNA capa transformer por posición: QKVO (4d^2) + FFN (2*ffn*d^2) + atención causal (~ctx/2)."""
    return 2 * (4 * d * d + 2 * ffn * d * d) + 4 * (ctx / 2) * d

def flops_por_byte(modelo, d_g=4096, L_g=32, ctx_bytes=8192, vocab_bpe=128_000, bpt=4.4, ps=4.5,
                   d_l=1024, L_enc=1, L_dec=6, ventana=512):
    """FLOPs de forward POR BYTE de texto. Config ILUSTRATIVA tipo 8B (NO es la del paper)."""
    if modelo == "bpe":      # un paso del transformer grande por token + proyección a un vocabulario grande
        return (L_g * flops_capa(d_g, ctx_bytes / bpt) + 2 * d_g * vocab_bpe) / bpt
    if modelo == "bytes":    # transformer isotrópico: un paso por byte, atención sobre todos los bytes
        return L_g * flops_capa(d_g, ctx_bytes) + 2 * d_g * 256
    if modelo == "blt":      # global (1 paso por parche) + modelos locales (1 paso por byte) + atenciones cruzadas
        glob = L_g * flops_capa(d_g, ctx_bytes / ps) / ps
        local = (L_enc + L_dec) * flops_capa(d_l, ventana) + 2 * d_l * 256
        cross = 4 * d_l * d_g
        return glob + local + cross
    raise ValueError(modelo)


def inyecta_erratas(texto, p, rng):
    """Sustituye UNA letra ASCII de cada palabra afectada (prob. p, palabras ASCII de >=4 letras). Longitud en bytes intacta.
    Devuelve (texto_ruidoso, [(ini, fin) de las palabras corrompidas en bytes])."""
    out, malas = list(texto), []
    for m in re.finditer(r"[A-Za-z]{4,}", texto):
        if rng.random() < p:
            i = rng.randrange(m.start(), m.end())
            out[i] = rng.choice([c for c in "abcdefghijklmnopqrstuvwxyz" if c != texto[i].lower()])
            malas.append((m.start(), m.end()))
    return "".join(out), malas

def palabras_ascii(texto):
    return [(m.start(), m.end()) for m in re.finditer(r"[A-Za-z]{4,}", texto)]

El modelo de bytes de orden 4 (Witten-Bell) es el de la lección; **su método `entropias` llama a tu función `entropia_bits`** (TODO 1).

In [ ]:
from collections import defaultdict

class NgramByteLM:
    """Modelo de bytes de orden n con suavizado Witten-Bell: una TABLA de conteos (el 'lookup table' que
    menciona el paper de BLT cuando el campo receptivo del modelo de entropía es pequeño)."""
    def __init__(self, orden=4):
        self.orden = orden
        self.cuentas = [defaultdict(lambda: np.zeros(256, np.int32)) for _ in range(orden + 1)]
    def fit(self, data):
        b = data.tobytes()
        for n in range(self.orden + 1):
            for i in range(n, len(b)):
                self.cuentas[n][b[i - n:i]][b[i]] += 1
        return self
    def probs(self, ctx: bytes):
        p = np.full(256, 1 / 256)
        for n in range(self.orden + 1):
            if n > len(ctx):
                break
            c = self.cuentas[n].get(ctx[len(ctx) - n:] if n else b"")
            if c is None:
                break
            N, T = c.sum(), (c > 0).sum()
            p = (c + T * p) / (N + T)                      # interpolación Witten-Bell con el orden inferior
        return p
    def entropias(self, data):
        """H[i] = entropía (bits) de la distribución del byte i dado x_<i. H[0] = 8 (sin contexto)."""
        b = data.tobytes(); H = np.zeros(len(b), np.float32)
        for i in range(len(b)):
            H[i] = entropia_bits(self.probs(b[max(0, i - self.orden):i]))   # <- tu TODO 1
        return H
    def bpb(self, data):
        b = data.tobytes(); tot = 0.0
        for i in range(len(b)):
            tot -= np.log2(self.probs(b[max(0, i - self.orden):i])[b[i]])
        return tot / len(b)

## Paso 1 · TODO 1: entropía de Shannon

$$H(p) = -\sum_{v} p_v \log_2 p_v$$

Solo se suman los $p_v>0$ (convención $0\log 0=0$).

In [ ]:
def entropia_bits(p):
    """Entropía de Shannon, en bits, de un vector de probabilidades p (256 posiciones; puede haber ceros)."""
    # TODO 1: H = - sum_v p_v log2 p_v ignorando p_v = 0
    raise NotImplementedError("TODO 1")

## Paso 2 · TODO 2-3: reglas de parcheo y calibración del umbral

In [ ]:
def inicios_global(H, theta):
    """H: array de entropías por byte. Devuelve bool[len(H)]: True si el byte ABRE un parche (regla global)."""
    # TODO 2a: H[i] > theta; el primer byte siempre abre parche
    raise NotImplementedError("TODO 2a")

def inicios_monotono(H, theta_r):
    """Regla de restricción monótona aproximada: H[i] - H[i-1] > theta_r (el primer byte siempre abre parche)."""
    # TODO 2b
    raise NotImplementedError("TODO 2b")

<details>
<summary>🪜 Pista 1</summary>

Regla global: `s = H > theta` y fuerza `s[0] = True`. Cuidado: no modifiques `H` en sitio (`s` es un array nuevo).

</details>

<details>
<summary>🪜 Pista 2</summary>

Regla monótona: construye `d` con `d[0] = np.inf` y `d[1:] = H[1:] - H[:-1]`; `s = d > theta_r`; `s[0] = True`.

</details>

In [ ]:
def calibra(H, objetivo, fn, lo=-6.0, hi=9.0, iters=40):
    """Busca por bisección el umbral theta tal que el tamaño medio de parche (len(H) / nº de parches) sea `objetivo`.
    `fn(H, theta)` devuelve los inicios. Pista de monotonía: a mayor umbral, menos parches, parches más largos."""
    # TODO 3
    raise NotImplementedError("TODO 3")

<details>
<summary>🪜 Pista 1</summary>

Bisección: en cada iteración `mid = (lo+hi)/2`, calcula `ps = len(H) / fn(H, mid).sum()`. Si `ps` es MAYOR que el objetivo hay pocos parches: baja el umbral (`hi = mid`); si no, sube (`lo = mid`).

</details>

<details>
<summary>🪜 Pista 2</summary>

`for _ in range(iters): mid = (lo+hi)/2; if len(H)/fn(H, mid).sum() > objetivo: hi = mid  else: lo = mid` y devuelve `(lo+hi)/2`. No olvides la función `longitudes` (dada en la solución) si la necesitas.

</details>

In [ ]:
# --- Flujo (no lo cambies): modelo de entropía + calibración de umbrales ---
RES = {}
try:
    NG4 = NgramByteLM(4).fit(TR)
    H_TR, H_VA = NG4.entropias(TR), NG4.entropias(VA)
    print("sanity: H(uniforme 256) =", entropia_bits(np.full(256, 1 / 256)), "bits (debe ser 8.0)")
    OBJ = 4.5
    TH_G = calibra(H_TR, OBJ, inicios_global); TH_M = calibra(H_TR, OBJ, inicios_monotono)
    ps_g = len(H_VA) / inicios_global(H_VA, TH_G).sum(); ps_m = len(H_VA) / inicios_monotono(H_VA, TH_M).sum()
    RES.update(th_g=TH_G, th_m=TH_M, ps_g=ps_g, ps_m=ps_m)
    print(f"θ_global={TH_G:.3f} -> ps(val)={ps_g:.2f} | θ_monótono={TH_M:.3f} -> ps(val)={ps_m:.2f}  (objetivo {OBJ})")
except NotImplementedError as e:
    print("Pendiente:", e)

> **👀 Qué debes observar**
>
> Los dos umbrales (global y monótono) son distintos pero producen el mismo tamaño medio de parche; la diferencia está en **dónde** cortan. Compáralos mirando las longitudes en la solución de referencia.

## Paso 3 · TODO 4: parcheo incremental

$f_p(\boldsymbol{x}_{<i}) = f_p(\boldsymbol{x})_{<i}$ — mide la fracción de cortes aleatorios en que se cumple.

In [ ]:
def prueba_incremental(f_inicios, items, n_pruebas=300, seed=SEED):
    """Fracción de cortes aleatorios i en los que f(x[:i]) == f(x)[:i]  (propiedad de parcheo incremental).
    `items`: lista de str (para BPE) o de bytes (para entropía); `f_inicios` acepta el mismo tipo."""
    # TODO 4: elige un item y un corte i; compara f(prefijo) con f(completo)[:len(prefijo)]
    raise NotImplementedError("TODO 4")

<details>
<summary>🪜 Pista 1</summary>

Usa `random.Random(seed)`; para cada prueba elige `it`, un corte `i` entre 5 y `len-1` y calcula `full = f(it)` y `pref = f(prefijo)`.

</details>

<details>
<summary>🪜 Pista 2</summary>

El prefijo de un `bytes` es `it[:i]`; el de un `str` es `it.encode()[:i].decode('utf-8','ignore')`. Compara `np.array_equal(full[:len(pref)], pref)` y promedia.

</details>

In [ ]:
# --- Flujo: parcheo incremental ---
try:
    textos_prueba = [d["text"][:300] for d in DOCS_VA if d["lang"] != "code"][:60]
    bytes_prueba = [t.encode() for t in textos_prueba]
    f_ent_b = lambda b: inicios_global(NG4.entropias(np.frombuffer(b, np.uint8)), TH_G)
    f_bpe_t = lambda t: inicios_bpe(TK, t)
    RES["inc_ent"] = prueba_incremental(f_ent_b, bytes_prueba)
    RES["inc_bpe"] = prueba_incremental(f_bpe_t, textos_prueba)
    print(f"incremental: entropía {RES['inc_ent']*100:.1f}% | BPE {RES['inc_bpe']*100:.1f}%")
except (NotImplementedError, NameError) as e:
    print("Pendiente:", e)

## Paso 4 · TODO 5: robustez a erratas

Inyectamos una errata (una letra cambiada) en el 5 % de las palabras. Dos preguntas: ¿**se infla** el número de unidades (= coste)? ¿**cambia** la segmentación de las palabras que *no* tocamos (= inestabilidad de contexto)?

In [ ]:
def metricas_robustez(f_seg, textos, p=0.05, seed=SEED):
    """Compara la segmentación de textos limpios y con erratas (usa `inyecta_erratas`).
    Devuelve {"frag": unidades_ruidoso / unidades_limpio,   # inflación del nº de parches/tokens (coste) por el ruido
              "estab": fracción de palabras INTACTAS cuya segmentación interna (bool por byte) no cambia}."""
    # TODO 5: f_seg(texto) -> bool por byte. Suma unidades sobre todos los textos; para 'estab' compara
    #         sc[a:b] con sn[a:b] en palabras (a, b) de `palabras_ascii(texto)` que NO estén en `malas`.
    raise NotImplementedError("TODO 5")

<details>
<summary>🪜 Pista 1</summary>

`sc = f_seg(t)`, `sn = f_seg(ruido)` tienen la misma longitud (la errata no cambia el nº de bytes). `frag = suma(sn) / suma(sc)` sobre todos los textos.

</details>

<details>
<summary>🪜 Pista 2</summary>

Para la estabilidad itera `for (a, b) in palabras_ascii(t)`; salta las que estén en `malas`; cuenta `np.array_equal(sc[a:b], sn[a:b])`. Devuelve `{'frag': ..., 'estab': ok/tot}`.

</details>

In [ ]:
L_BPE = np.concatenate([[len(t) for t in TK.encode(d['text'][:MAX_CHARS]).tokens] for d in DOCS_VA])   # bytes por token (val)

# --- Flujo: robustez a erratas (5 % de las palabras) ---
try:
    textos_rob = [d["text"][:400] for d in DOCS_VA if d["lang"] in ("es", "en")][:40]
    seg_ent = lambda t: inicios_global(NG4.entropias(np.frombuffer(t.encode(), np.uint8)), TH_G)
    seg_bpe = lambda t: inicios_bpe(TK, t)
    RES["rob_ent"] = metricas_robustez(seg_ent, textos_rob)
    RES["rob_bpe"] = metricas_robustez(seg_bpe, textos_rob)
    RES["ps_bpe"] = float(np.mean(L_BPE)) if "L_BPE" in globals() else None
    print("entropía:", {k: round(v, 3) for k, v in RES["rob_ent"].items()})
    print("BPE     :", {k: round(v, 3) for k, v in RES["rob_bpe"].items()})
except (NotImplementedError, NameError) as e:
    print("Pendiente:", e)

## Paso 5 · TODO 6: coste

Con la fórmula de FLOPs/byte de la lección (config. ilustrativa tipo 8B), ¿qué tamaño de parche necesita BLT para gastar ≥ 25 % menos que un BPE industrial (4.4 bytes/token, vocabulario de 128K)? (Tu BPE de 512 tokens es mucho más pequeño que el industrial: por eso la referencia se fija a 4.4.)

In [ ]:
def ps_para_ahorro(ahorro, bpt, grid=np.arange(3.0, 16.0, 0.25)):
    """Menor tamaño medio de parche `ps` para el que BLT gasta al menos `ahorro` (p. ej. 0.25 = 25 %) menos FLOPs/byte que BPE
    con `bpt` bytes/token. Usa flops_por_byte("blt", ps=...) y flops_por_byte("bpe", bpt=...)."""
    # TODO 6
    raise NotImplementedError("TODO 6")

<details>
<summary>🪜 Pista 1</summary>

Recorre `grid` de menor a mayor `ps` y devuelve el primero que cumpla `flops_por_byte('blt', ps=ps) <= (1-ahorro) * flops_por_byte('bpe', bpt=bpt)`.

</details>

In [ ]:
# --- Flujo: ¿qué parche medio necesito para ahorrar un 25 % de FLOPs/byte frente a un BPE "industrial"? ---
try:
    BPT_IND = 4.4          # bytes/token de Llama 3 (BLT, §2.4); la referencia usa un vocabulario de 128K en flops_por_byte("bpe")
    RES["bpt_propio"] = len(TXT_TR.encode()) / len(TK.encode(TXT_TR).ids)
    RES["ps_25"] = ps_para_ahorro(0.25, BPT_IND)
    print(f"tu BPE de {TK.get_vocab_size()} tokens da {RES['bpt_propio']:.2f} bytes/token (un vocabulario de 128K da ~4.4)")
    print(f"ps mínimo para -25% FLOPs/byte frente a BPE industrial (4.4 B/token, config ilustrativa tipo 8B): {RES['ps_25']:.2f}")
except (NotImplementedError, NameError) as e:
    print("Pendiente:", e)

In [ ]:
# --- Figura de resumen (no la cambies) ---
if all(k in RES for k in ["rob_ent", "rob_bpe", "inc_ent"]):
    fig, axs = plt.subplots(1, 3, figsize=(12, 3.6))
    axs[0].bar(["entropía", "BPE"], [RES["inc_ent"], RES["inc_bpe"]], color=[AZUL, NARANJA]); axs[0].set_ylim(0, 1.05); axs[0].set_title("Parcheo incremental (fracción de cortes)")
    axs[1].bar(["entropía", "BPE"], [RES["rob_ent"]["estab"], RES["rob_bpe"]["estab"]], color=[AZUL, NARANJA]); axs[1].set_ylim(0, 1.05); axs[1].set_title("Palabras intactas igual segmentadas")
    axs[2].bar(["entropía", "BPE"], [RES["rob_ent"]["frag"], RES["rob_bpe"]["frag"]], color=[AZUL, NARANJA]); axs[2].axhline(1, color=GRIS, ls=":"); axs[2].set_title("Inflación de unidades con 5 % de erratas")
    plt.tight_layout(); plt.show()
else:
    print("Completa los TODO 1-5 para ver la figura.")

> **👀 Qué debes observar**
>
> **Izquierda:** la barra azul (entropía) debe estar en 1.0 —el parcheo es incremental por construcción— y la naranja (BPE) por debajo: cortar el texto a mitad cambia la tokenización del prefijo. **Centro:** cuanto más cerca de 1.0, menos se alteran las palabras que la errata *no* tocó. **Derecha:** por encima de la línea punteada (1.0) la errata **encarece** la secuencia. Si una barra contradice lo que viste en la lección, revisa primero tu `entropia_bits` (TODO 1): todo lo demás depende de ella.

## Paso 6 · Conclusión y rúbrica

Escribe tu recomendación en `CONCLUSION` (≥ 300 caracteres): ¿qué ventajas/limitaciones ves frente a BPE? ¿qué **no** puedes concluir con este banco? (pista: no hemos entrenado un LM de parches en este proyecto, ni usamos datos reales ni varias semillas).

In [ ]:
CONCLUSION = ""   # TODO 7: escribe aquí tu nota técnica para el CTO (>= 300 caracteres)

In [ ]:
THR = {'ps_obj': 4.5, 'ps_tol': 0.1, 'frag_ent_max': 1.1, 'estab_ent_min': 0.95, 'estab_bpe_min': 0.98, 'ahorro_ps_max': 6.5}

def rubrica(RES, th=THR):
    filas = []
    def chk(nombre, ok, detalle): filas.append((nombre, ok, detalle))
    g = lambda k: RES.get(k)
    chk("R1 tamaño medio de parche (global y monótono) dentro de ±%d%% del objetivo" % (th["ps_tol"] * 100),
        g("ps_g") is not None and abs(g("ps_g") - 4.5) / 4.5 <= th["ps_tol"] and abs(g("ps_m") - 4.5) / 4.5 <= th["ps_tol"],
        f"ps_g={g('ps_g')}, ps_m={g('ps_m')}")
    chk("R2 parcheo incremental por entropía = 100 % y BPE < 100 %",
        g("inc_ent") is not None and g("inc_ent") == 1.0 and g("inc_bpe") < 1.0, f"ent={g('inc_ent')}, bpe={g('inc_bpe')}")
    re_, rb = g("rob_ent") or {}, g("rob_bpe") or {}
    chk("R3 inflación con erratas (entropía) <= %.2f" % th["frag_ent_max"], re_.get("frag", 9) <= th["frag_ent_max"], f"{re_}")
    chk("R4 estabilidad de palabras intactas (entropía) >= %.2f" % th["estab_ent_min"], re_.get("estab", 0) >= th["estab_ent_min"], f"{re_}")
    chk("R5 estabilidad de palabras intactas (BPE) >= %.2f (sanity de tu métrica)" % th["estab_bpe_min"], rb.get("estab", 0) >= th["estab_bpe_min"], f"{rb}")
    chk("R6 ps mínimo para -25%% FLOPs/byte <= %.1f" % th["ahorro_ps_max"], g("ps_25") is not None and g("ps_25") <= th["ahorro_ps_max"], f"ps_25={g('ps_25')}")
    chk("R7 conclusión escrita (>= 300 caracteres) en CONCLUSION", len(globals().get("CONCLUSION", "")) >= 300, f"{len(globals().get('CONCLUSION', ''))} caracteres")
    for n, ok, d in filas:
        print(("✅" if ok else "❌"), n, "|", d)
    print(f"\n{sum(o for _, o, _ in filas)}/{len(filas)} criterios")

rubrica(RES)

---
## ⛔ SPOILER — intenta resolverlo primero

Debajo está la solución de referencia **completa y ejecutable**. Si la ejecutas, redefine todas las funciones y rellena `RES` y `CONCLUSION`.
Si tu versión ya pasa la rúbrica, compara estilo y casos borde (ceros en `p`, prefijos vacíos, palabras con acentos).

### Solución de referencia

In [ ]:
def entropia_bits(p):
    p = np.asarray(p, dtype=np.float64); p = p[p > 0]
    return float(-(p * np.log2(p)).sum())

In [ ]:
def inicios_global(H, theta):                    # BLT: nuevo parche si H(x_i) > theta
    s = H > theta; s[0] = True; return s

def inicios_monotono(H, theta_r):                # BLT: nuevo parche si H(x_i) - H(x_{i-1}) > theta_r
    d = np.empty_like(H); d[0] = np.inf; d[1:] = H[1:] - H[:-1]
    s = d > theta_r; s[0] = True; return s

In [ ]:
def calibra(H, objetivo, fn, lo=-6.0, hi=9.0):
    """Bisección del umbral para que el tamaño medio de parche (bytes/parche) sea `objetivo`."""
    for _ in range(40):
        mid = (lo + hi) / 2
        if len(H) / fn(H, mid).sum() > objetivo:
            hi = mid
        else:
            lo = mid
    return (lo + hi) / 2


def longitudes(starts):
    idx = np.flatnonzero(starts); return np.diff(np.r_[idx, len(starts)])

In [ ]:
def prueba_incremental(fn_inicios, items, n_pruebas=300, seed=SEED):
    """fp(x_<i) == fp(x)_<i ? Devuelve la fracción de cortes en los que se cumple."""
    r = random.Random(seed); ok = 0
    for _ in range(n_pruebas):
        it = r.choice(items); n = len(it if isinstance(it, bytes) else it.encode())
        i = r.randrange(5, n - 1)
        full, pref = fn_inicios(it), fn_inicios(it[:i] if isinstance(it, bytes) else it.encode()[:i].decode("utf-8", "ignore"))
        ok += int(np.array_equal(full[:len(pref)], pref))
    return ok / n_pruebas

In [ ]:
def metricas_robustez(f_seg, textos, p=0.05, seed=SEED):
    r = random.Random(seed); u_c = u_n = ok = tot = 0
    for t in textos:
        ruido, malas = inyecta_erratas(t, p, r)
        sc, sn = f_seg(t), f_seg(ruido)
        u_c += int(sc.sum()); u_n += int(sn.sum())
        for (a, b) in palabras_ascii(t):
            if (a, b) not in malas and b <= min(len(sc), len(sn)):
                tot += 1; ok += int(np.array_equal(sc[a:b], sn[a:b]))
    return {"frag": u_n / u_c, "estab": ok / max(1, tot)}

In [ ]:
def ps_para_ahorro(ahorro, bpt, grid=np.arange(3.0, 16.0, 0.25)):
    ref = flops_por_byte("bpe", bpt=bpt)
    for ps in grid:
        if flops_por_byte("blt", ps=float(ps)) <= (1 - ahorro) * ref:
            return float(ps)
    return float("nan")

In [ ]:
# --- Flujo (no lo cambies): modelo de entropía + calibración de umbrales ---
RES = {}
try:
    NG4 = NgramByteLM(4).fit(TR)
    H_TR, H_VA = NG4.entropias(TR), NG4.entropias(VA)
    print("sanity: H(uniforme 256) =", entropia_bits(np.full(256, 1 / 256)), "bits (debe ser 8.0)")
    OBJ = 4.5
    TH_G = calibra(H_TR, OBJ, inicios_global); TH_M = calibra(H_TR, OBJ, inicios_monotono)
    ps_g = len(H_VA) / inicios_global(H_VA, TH_G).sum(); ps_m = len(H_VA) / inicios_monotono(H_VA, TH_M).sum()
    RES.update(th_g=TH_G, th_m=TH_M, ps_g=ps_g, ps_m=ps_m)
    print(f"θ_global={TH_G:.3f} -> ps(val)={ps_g:.2f} | θ_monótono={TH_M:.3f} -> ps(val)={ps_m:.2f}  (objetivo {OBJ})")
except NotImplementedError as e:
    print("Pendiente:", e)


# --- Flujo: parcheo incremental ---
try:
    textos_prueba = [d["text"][:300] for d in DOCS_VA if d["lang"] != "code"][:60]
    bytes_prueba = [t.encode() for t in textos_prueba]
    f_ent_b = lambda b: inicios_global(NG4.entropias(np.frombuffer(b, np.uint8)), TH_G)
    f_bpe_t = lambda t: inicios_bpe(TK, t)
    RES["inc_ent"] = prueba_incremental(f_ent_b, bytes_prueba)
    RES["inc_bpe"] = prueba_incremental(f_bpe_t, textos_prueba)
    print(f"incremental: entropía {RES['inc_ent']*100:.1f}% | BPE {RES['inc_bpe']*100:.1f}%")
except (NotImplementedError, NameError) as e:
    print("Pendiente:", e)

In [ ]:
# --- Flujo: robustez a erratas (5 % de las palabras) ---
try:
    textos_rob = [d["text"][:400] for d in DOCS_VA if d["lang"] in ("es", "en")][:40]
    seg_ent = lambda t: inicios_global(NG4.entropias(np.frombuffer(t.encode(), np.uint8)), TH_G)
    seg_bpe = lambda t: inicios_bpe(TK, t)
    RES["rob_ent"] = metricas_robustez(seg_ent, textos_rob)
    RES["rob_bpe"] = metricas_robustez(seg_bpe, textos_rob)
    RES["ps_bpe"] = float(np.mean(L_BPE)) if "L_BPE" in globals() else None
    print("entropía:", {k: round(v, 3) for k, v in RES["rob_ent"].items()})
    print("BPE     :", {k: round(v, 3) for k, v in RES["rob_bpe"].items()})
except (NotImplementedError, NameError) as e:
    print("Pendiente:", e)


# --- Flujo: ¿qué parche medio necesito para ahorrar un 25 % de FLOPs/byte frente a un BPE "industrial"? ---
try:
    BPT_IND = 4.4          # bytes/token de Llama 3 (BLT, §2.4); la referencia usa un vocabulario de 128K en flops_por_byte("bpe")
    RES["bpt_propio"] = len(TXT_TR.encode()) / len(TK.encode(TXT_TR).ids)
    RES["ps_25"] = ps_para_ahorro(0.25, BPT_IND)
    print(f"tu BPE de {TK.get_vocab_size()} tokens da {RES['bpt_propio']:.2f} bytes/token (un vocabulario de 128K da ~4.4)")
    print(f"ps mínimo para -25% FLOPs/byte frente a BPE industrial (4.4 B/token, config ilustrativa tipo 8B): {RES['ps_25']:.2f}")
except (NotImplementedError, NameError) as e:
    print("Pendiente:", e)

In [ ]:
# --- Figura de resumen (no la cambies) ---
if all(k in RES for k in ["rob_ent", "rob_bpe", "inc_ent"]):
    fig, axs = plt.subplots(1, 3, figsize=(12, 3.6))
    axs[0].bar(["entropía", "BPE"], [RES["inc_ent"], RES["inc_bpe"]], color=[AZUL, NARANJA]); axs[0].set_ylim(0, 1.05); axs[0].set_title("Parcheo incremental (fracción de cortes)")
    axs[1].bar(["entropía", "BPE"], [RES["rob_ent"]["estab"], RES["rob_bpe"]["estab"]], color=[AZUL, NARANJA]); axs[1].set_ylim(0, 1.05); axs[1].set_title("Palabras intactas igual segmentadas")
    axs[2].bar(["entropía", "BPE"], [RES["rob_ent"]["frag"], RES["rob_bpe"]["frag"]], color=[AZUL, NARANJA]); axs[2].axhline(1, color=GRIS, ls=":"); axs[2].set_title("Inflación de unidades con 5 % de erratas")
    plt.tight_layout(); plt.show()
else:
    print("Completa los TODO 1-5 para ver la figura.")

In [ ]:
CONCLUSION = (
    "Recomendación: no sustituir BPE todavía para el generativo de Políglota; sí seguir de cerca los parches por entropía. "
    f"A tamaño de unidad similar (~4.5 bytes), el parcheo por entropía es incremental por construcción ({RES['inc_ent']*100:.0f} %), mientras que BPE no lo es ({RES['inc_bpe']*100:.0f} %). "
    f"Ante un 5 % de erratas la segmentación por entropía infla las unidades un {100*(RES['rob_ent']['frag']-1):.1f} % (BPE: {100*(RES['rob_bpe']['frag']-1):.1f} %) y mantiene la segmentación del "
    f"{RES['rob_ent']['estab']*100:.1f} % de las palabras intactas (BPE con pre-tokenización: {RES['rob_bpe']['estab']*100:.1f} %): la dependencia del contexto previo es real pero pequeña aquí. "
    f"Para ahorrar un 25 % de FLOPs/byte frente a un BPE industrial haría falta un parche medio de ~{RES['ps_25']:.1f} bytes. "
    "Limitaciones: modelo de entropía n-grama, corpus pequeño/sintético, una semilla, sin entrenar el LM de parches ni medir bpb ni latencia; "
    "los FLOPs son una fórmula ilustrativa. Siguiente paso: repetir con datos reales y semillas, y medir bpb con un MiniBLT a FLOPs iguales."
)
rubrica(RES)

## Retos extra (nivel experto)

1. Sustituye la tabla n-grama por el **transformer diminuto de la lección** como modelo de entropía y compara F1 de fronteras con las del n-grama.
2. Implementa el **reinicio del contexto en saltos de línea** (BLT §4.4) y mide cuánto cambia el tamaño medio de parche en texto con mucha repetición.
3. Prueba **orden 2, 3, 5 y 6** del n-grama: ¿cómo cambian el bpb del modelo de entropía y la calidad de las fronteras (precisión/recall frente a inicios de palabra)?
4. Mide la estabilidad con **cambios de mayúsculas** y con **espacios dobles** en vez de erratas: ¿qué segmentador sufre más?
5. Escribe un `ParchesEntropicosDataset` que devuelva (bytes, inicios) listos para alimentar el MiniBLT de la lección.

## Reflexión

* ¿Qué significa para un servicio de búsqueda como Políglota que la segmentación de una palabra **dependa del contexto previo**? (cachés, deduplicación, índices léxicos).
* ¿Qué parte de la fragilidad ante erratas que mediste en el módulo 05 es culpa del **vocabulario** y cuál de la **segmentación**?
* ¿Qué experimento harías, con presupuesto de una semana de GPU, para decidir si BLT merece entrar en la hoja de ruta de Políglota?
* ¿Cómo se conecta con el módulo 07? (la frontera de parche pasa de heurística externa a algo que se **entrena**).